# PEMSD8 physics-loss rerun: authors' own code, new seeds

**Question.** Does the paper's new contribution, the Underwood physics loss, beat run-to-run noise when we use **the authors' own physics-loss code** and **new random seeds**?

The first run (findings 40–42) paired our GPDE+Quantile seeds 0–2 with physics runs that used a re-implemented loss. This rerun removes both doubts:

| Run label | Model | Code | Seed |
|---|---|---|---|
| `pemsd8_gpde_quantile_seed3` | GPDE+Quantile | `gpde_quantile_model_PEMSD8` | 3 |
| `pemsd8_gpde_quantile_phy_seed3` | GPDE+Quantile+Phy (λ = 0.002) | `gpde_quantile_phy_model_PEMSD8`, the authors' own loss | 3 |
| `pemsd8_gpde_quantile_seed4` | GPDE+Quantile | `gpde_quantile_model_PEMSD8` | 4 |
| `pemsd8_gpde_quantile_phy_seed4` | GPDE+Quantile+Phy (λ = 0.002) | `gpde_quantile_phy_model_PEMSD8` | 4 |

Same seed = same starting weights for both models, so each seed gives one paired comparison. Section 5 reports the two new pairs on their own, then pooled with the first run's three pairs (five pairs in total).


## 0. Drive, settings, and which runs this account does

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, glob, re, shutil, subprocess, threading, time
READ_BASE  = '/content/drive/MyDrive/CS637'          # read only: authors' code, first-run results
WRITE_BASE = '/content/drive/MyDrive/CS637_rerun'    # everything this notebook writes
for b in [READ_BASE, WRITE_BASE]:
    assert os.path.isdir(b), f'{b} not found: see "Before you start" above (folder or shortcut missing)'
BASE  = READ_BASE
AUTH  = f'{READ_BASE}/uq4phy_traffic_network'
P1    = f'{WRITE_BASE}/p1'
RUNDR = f'{P1}/pemsd8_runs'              # checkpoints, logs and DONE markers
OUT8  = f'{WRITE_BASE}/outputs/pemsd8'   # prediction files (.npz), only if SAVE_PREDICTIONS
RES   = f'{P1}/results'
DATA8 = '/content/data/PEMSD8'           # local disk, downloaded every session
SCRIPTS = '/content/p1_scripts'         # helper scripts run from local disk
for d in [RUNDR, OUT8, RES, SCRIPTS, f'{RES}/pemsd8_runs']:
    os.makedirs(d, exist_ok=True)

RUNS = {  # label -> (authors' folder, seed)
    'pemsd8_gpde_quantile_seed3':     ('gpde_quantile_model_PEMSD8', 3),
    'pemsd8_gpde_quantile_phy_seed3': ('gpde_quantile_phy_model_PEMSD8', 3),
    'pemsd8_gpde_quantile_seed4':     ('gpde_quantile_model_PEMSD8', 4),
    'pemsd8_gpde_quantile_phy_seed4': ('gpde_quantile_phy_model_PEMSD8', 4),
}
# ---- choose per account ----
# account A: MY_RUNS = ['pemsd8_gpde_quantile_seed3', 'pemsd8_gpde_quantile_phy_seed3']
# account B: MY_RUNS = ['pemsd8_gpde_quantile_seed4', 'pemsd8_gpde_quantile_phy_seed4']
MY_RUNS = ['pemsd8_gpde_quantile_seed3', 'pemsd8_gpde_quantile_phy_seed3']
SAVE_PREDICTIONS = False  # True keeps about 120 MB per run in CS637_rerun/outputs/pemsd8/; the analysis doesn't need them

# FRESH_START = True deletes THIS account's runs (MY_RUNS) from CS637_rerun and retrains them.
# Use it for the first Run all. If the session dies and you run again, set it to False first,
# or finished runs get deleted and retrained.
FRESH_START = True
CLEARED = set()
if FRESH_START:
    for r in MY_RUNS:
        for path in [f'{RUNDR}/{r}', f'{RES}/pemsd8_runs/{r}.csv', f'{OUT8}/{r}.npz']:
            try:
                shutil.rmtree(path) if os.path.isdir(path) else (os.remove(path) if os.path.exists(path) else None)
            except OSError as e:
                print('note: could not delete', path, e)
        CLEARED.add(r)
    print('FRESH_START: cleared', sorted(CLEARED), '- these will be trained from scratch')
for r in RUNS:
    print(f'{r:34s}', 'DONE' if os.path.exists(f'{RUNDR}/{r}/DONE') else '-', '<- this account' if r in MY_RUNS else '')

Mounted at /content/drive
FRESH_START: cleared ['pemsd8_gpde_quantile_phy_seed3', 'pemsd8_gpde_quantile_seed3'] - these will be trained from scratch
pemsd8_gpde_quantile_seed3         - <- this account
pemsd8_gpde_quantile_phy_seed3     - <- this account
pemsd8_gpde_quantile_seed4         - 
pemsd8_gpde_quantile_phy_seed4     - 


In [2]:
import torch, numpy as np
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv

torch 2.11.0+cu130 | cuda True
name, memory.used [MiB], memory.total [MiB]
Tesla T4, 3 MiB, 15360 MiB


## 1. Authors' PEMSD8 folders

Notebook 1 already cloned these three folders. This cell only checks they're there.

In [3]:
for f in sorted({v[0] for v in RUNS.values()}):
    ok = os.path.exists(f'{AUTH}/{f}/dcrnn_train_pytorch.py')
    print(f'{f:32s}', 'ok' if ok else 'MISSING')
    if not ok:
        !git -C "{AUTH}" sparse-checkout add /{f}/

gpde_quantile_model_PEMSD8       ok
gpde_quantile_phy_model_PEMSD8   ok


## 2. PEMSD8 data (every session, about 2 minutes)

These are the authors' processed files (train/val/test.npz) from figshare article 28347662, saved to Colab's local disk.
Writing them to the shared Drive folder fails part-way and uses up the uploader's quota.

In [4]:
import requests, zipfile
def get_figshare(article_id, dst):
    os.makedirs(dst, exist_ok=True)
    files = requests.get(f'https://api.figshare.com/v2/articles/{article_id}', timeout=60).json()['files']
    for f in files:
        path = f'{dst}/{f["name"]}'
        if not (os.path.exists(path) and os.path.getsize(path) == f['size']):
            print('downloading', f['name'], f'{f["size"]/1e6:.0f} MB')
            with requests.get(f['download_url'], stream=True, timeout=600) as r:
                r.raise_for_status()
                with open(path, 'wb') as out:
                    for chunk in r.iter_content(1 << 22): out.write(chunk)
        if path.endswith('.zip'):
            zipfile.ZipFile(path).extractall(dst)
    return dst

get_figshare(28347662, DATA8)
found = {s: sorted(glob.glob(f'{DATA8}/**/{s}.npz', recursive=True)) for s in ['train', 'val', 'test']}
DATA8 = os.path.dirname(found['test'][0])            # in case the files sit in a subfolder
shapes = {s: np.load(f'{DATA8}/{s}.npz')['x'].shape for s in ['train', 'val', 'test']}
print('DATA8 =', DATA8, shapes)
assert all(v[2:] == (170, 3) for v in shapes.values()), 'expected (N, 12, 170, 3)'

downloading train.npz 611 MB
downloading PEMSD8 data processing.py 0 MB
downloading pems8ndy.npy 73 MB
downloading val.npz 86 MB
downloading test.npz 174 MB
DATA8 = /content/data/PEMSD8 {'train': (12476, 12, 170, 3), 'val': (1762, 12, 170, 3), 'test': (3549, 12, 170, 3)}


## 3. Helper scripts (updated versions of the notebook 1 scripts)

Written to Colab's local disk every session (`/content/p1_scripts`) and copied to `p1/scripts/` on Drive as a record.

Changes from notebook 1:
- `eval_checkpoint.py` takes `--config`, and re-enables the prediction output that the PEMSD8 variants comment out in `evaluate()`.
- Both scripts treat |truth| < 1e-3 as missing (finding 29).
- `horizon_metrics.py` can add the authors' batch padding (`pad_to_batch`) to match their reported numbers exactly.

Numbers from notebook 1 don't change: PEMSD4 has no missing readings.

In [5]:
%%writefile {SCRIPTS}/eval_checkpoint.py
"""Evaluate one of the authors' shipped checkpoints and save val + test predictions.

Run from INSIDE a variant folder of the authors' repo, e.g.
    cd .../uq4phy_traffic_network/gpde_quantile_phy_model_PEMSD4
    python /path/eval_checkpoint.py --data_dir /content/pemsd4 --out /path/model.npz
The authors' code is not modified: we load their pretrained config with yaml.safe_load
(their run_demo uses yaml.load(f), which fails on PyYAML 6) and let torch.load read
their full checkpoint dict (PyTorch >= 2.6 defaults to weights_only=True).
"""
import argparse, glob, json, os, sys, time
import numpy as np, torch, yaml

_torch_load = torch.load
def _load(*a, **k):
    k.setdefault('weights_only', False)
    return _torch_load(*a, **k)
torch.load = _load

sys.path.insert(0, os.getcwd())
from lib.utils import load_graph_data                      # noqa: E402  (authors' code)
from model.pytorch.dcrnn_supervisor import DCRNNSupervisor  # noqa: E402
import inspect, textwrap                                    # noqa: E402
import model.pytorch.dcrnn_supervisor as _sup               # noqa: E402

# In the PEMSD8 variants evaluate() builds the predictions but its return line ends in
# "loss_mae#,{'prediction': ...}" (commented out). Re-enable it here, without editing their file.
_src = inspect.getsource(DCRNNSupervisor.evaluate)
if "#,{'prediction'" in _src:
    _ns = {}
    exec(textwrap.dedent(_src.replace("#,{'prediction'", ",{'prediction'")), _sup.__dict__, _ns)
    DCRNNSupervisor.evaluate = _ns['evaluate']
    print('NOTE: re-enabled the prediction output of evaluate() (commented out in this variant)')

p = argparse.ArgumentParser()
p.add_argument('--data_dir', required=True)
p.add_argument('--out', required=True)
p.add_argument('--log_dir', default='/content/p1_eval_logs')
p.add_argument('--epoch', type=int, default=None, help='default: epoch in the pretrained config')
p.add_argument('--config', default=None, help='default: data/model/pretrained/*/config.yaml (shipped checkpoints)')
a = p.parse_args()

cfg_path = a.config or sorted(glob.glob('data/model/pretrained/*/config.yaml'))[0]
cfg = yaml.safe_load(open(cfg_path))
saved = sorted(int(f[3:-4]) for f in os.listdir('models') if f.startswith('epo') and f.endswith('.tar'))
epoch = a.epoch if a.epoch is not None else cfg['train']['epoch']
assert epoch in saved, f'epo{epoch}.tar not in models/ (have {saved})'
if epoch != max(saved):
    print(f'NOTE: config epoch {epoch} is not the latest saved checkpoint ({max(saved)})')
cfg['data']['dataset_dir'] = a.data_dir
cfg['train']['epoch'] = epoch
cfg['train']['log_dir'] = os.path.join(a.log_dir, os.path.basename(os.getcwd()))

_, _, adj = load_graph_data(cfg['data']['graph_pkl_filename'])
t0 = time.time()
sup = DCRNNSupervisor(adj_mx=adj, **cfg)
result, meta = {}, {'variant': os.path.basename(os.getcwd()), 'epoch': epoch, 'config': cfg_path,
                    'reg_lambda': cfg['model'].get('reg'), 'seed': os.environ.get('SEED'),
                    'test_batch_size': cfg['data'].get('test_batch_size'),
                    'device': str(next(sup.dcrnn_model.parameters()).device)}
for split in ['val', 'test']:
    mis, width, mse, rmse, mae, out = sup.evaluate(split)
    n = len(np.load(os.path.join(a.data_dir, split + '.npz'))['y'])
    pred = np.stack(out['prediction']).astype(np.float32)   # (12, N_padded, nodes, 3) = lower, median, upper
    truth = np.stack(out['truth']).astype(np.float32)       # (12, N_padded, nodes)
    near0 = (truth != 0) & (np.abs(truth) < 1e-3)           # missing readings come back as float residues
    meta[f'{split}_missing_share'] = float((np.abs(truth) < 1e-3).mean())
    truth[near0] = 0.0
    meta[f'{split}_samples'] = n
    meta[f'{split}_padded_samples'] = int(pred.shape[1])
    meta[f'{split}_authors_metrics'] = dict(mae=float(mae), mis=float(mis), width=float(width), mse=float(mse), rmse=float(rmse))
    result[f'{split}_pred'] = pred[:, :n]        # drop the copies of the last sample used to pad the final batch
    result[f'{split}_truth'] = truth[:, :n]
    print(f'{split}: authors-style (all 12 horizons, incl. padding)  MAE {mae:.4f}  MIS {mis:.4f}  '
          f'width {width:.4f}  MSE {mse:.4f}  RMSE {rmse:.4f}   [{n} samples, padded to {pred.shape[1]}]')
meta['seconds'] = round(time.time() - t0, 1)
os.makedirs(os.path.dirname(a.out), exist_ok=True)
np.savez_compressed(a.out, meta=json.dumps(meta), **result)
print('saved', a.out, f'({os.path.getsize(a.out)/1e6:.0f} MB, {meta["seconds"]} s)')

Writing /content/p1_scripts/eval_checkpoint.py


In [6]:
%%writefile {SCRIPTS}/horizon_metrics.py
"""Per-horizon metrics from the prediction files written by eval_checkpoint.py.

Two readings of "15 / 30 / 60 min", because the paper does not say which it uses:
  step : the single forecast step at that lead time (step 3, 6, 12 of 12)
  avg  : the average over all steps up to that lead time (steps 1-3, 1-6, 1-12);
         avg at 60 min is the all-horizon average the authors' code prints
MIS is reported two ways:
  mis_authors : the code's mis_loss = (u - l) + |y - median| + 40*(y - u)+ + 40*(l - y)+
  mis_std     : the paper's Eq. 4 / standard interval score, without the |y - median| term
Readings where truth == 0 are excluded, as in the authors' masked losses; |truth| < 1e-3 counts
as 0 too, because missing readings come back from evaluate() as tiny float residues.
pad_to_batch: add copies of the last sample the way the authors' DataLoader pads the final
batch (their reported numbers include these copies); use it only to match their numbers.
"""
import numpy as np

LEADS = {15: 3, 30: 6, 60: 12}           # minutes -> number of 5-minute steps
RHO = 2.0 / 0.05                         # 2 / alpha for a 95% interval

def _metrics(pred, truth):
    m = np.abs(truth) >= 1e-3
    lo, mid, hi, y = pred[..., 0][m], pred[..., 1][m], pred[..., 2][m], truth[m]
    lo, mid, hi, y = (v.astype(np.float64) for v in (lo, mid, hi, y))
    err = mid - y
    pen = RHO * np.maximum(y - hi, 0) + RHO * np.maximum(lo - y, 0)
    mse = np.mean(err ** 2)
    return dict(mae=np.mean(np.abs(err)), mse=mse, rmse=np.sqrt(mse),
                mis_authors=np.mean(hi - lo + np.abs(err) + pen), mis_std=np.mean(hi - lo + pen),
                width=np.mean(hi - lo), coverage=np.mean((y >= lo) & (y <= hi)),
                crossed=np.mean((lo > mid) | (mid > hi)))

def pad_like_authors(pred, truth, batch_size):
    n = pred.shape[1]; extra = (-n) % batch_size
    if extra == 0:
        return pred, truth
    return (np.concatenate([pred, np.repeat(pred[:, -1:], extra, axis=1)], axis=1),
            np.concatenate([truth, np.repeat(truth[:, -1:], extra, axis=1)], axis=1))

def horizon_table(pred, truth, pad_to_batch=None):
    """pred (12, N, nodes, 3), truth (12, N, nodes) -> list of dict rows."""
    if pad_to_batch:
        pred, truth = pad_like_authors(pred, truth, pad_to_batch)
    rows = []
    for minutes, k in LEADS.items():
        rows.append(dict(view='step', minutes=minutes, **_metrics(pred[k - 1], truth[k - 1])))
        rows.append(dict(view='avg', minutes=minutes, **_metrics(pred[:k], truth[:k])))
    return rows

Writing /content/p1_scripts/horizon_metrics.py


In [7]:
# keep a copy on Drive for the repo (best effort; the notebook itself uses the local copies)
for f in ['eval_checkpoint.py', 'horizon_metrics.py']:
    try:
        os.makedirs(f'{P1}/scripts', exist_ok=True); shutil.copy2(f'{SCRIPTS}/{f}', f'{P1}/scripts/{f}')
    except OSError as e:
        print('note: could not copy', f, 'to Drive yet:', e)
print('scripts ready in', SCRIPTS, os.listdir(SCRIPTS))

scripts ready in /content/p1_scripts ['horizon_metrics.py', 'eval_checkpoint.py']


## 4. Train and evaluate this account's runs

For each run, this cell:
1. Copies the authors' folder to Colab's local disk, without their checkpoints and logs.
2. Changes one line in our copy, `random_seed = 0` → the run's seed, and points the config at the local data.
3. Trains 50 epochs and prints one line per epoch. Every 2 minutes it copies checkpoints and logs to `CS637_rerun/p1/pemsd8_runs/<label>/` on Drive.
4. Marks the run `DONE`, then evaluates its best checkpoint (the last one saved, since a checkpoint is saved only when validation MAE improves) and writes per-horizon metrics.


In [8]:
sys.path.insert(0, SCRIPTS)
import importlib, horizon_metrics; importlib.reload(horizon_metrics)
import pandas as pd

def _epochs(d):
    return sorted(int(re.search(r'epo(\d+)\.tar', f).group(1)) for f in glob.glob(f'{d}/epo*.tar'))

def _copy_authors(folder, dst):
    shutil.rmtree(dst, ignore_errors=True)
    shutil.copytree(f'{AUTH}/{folder}', dst, ignore=shutil.ignore_patterns('models', 'dcrnn_DR_*', '__pycache__', '*.npz'))

def _retry(fn, what, tries=12, wait=15):
    # Drive (especially a shared folder used through a shortcut) can take a minute or two
    # before a new folder accepts files, so file operations on Drive are retried.
    last = None
    for _ in range(tries):
        try:
            return fn()
        except OSError as e:
            last = e; time.sleep(wait)
    print(f'WARNING: {what} still failing after {tries} tries: {last}')

def train(label):
    folder, seed = RUNS[label]
    ddir = f'{RUNDR}/{label}'
    if label not in CLEARED and os.path.exists(f'{ddir}/DONE'):
        print(f'{label}: already DONE on Drive, not retraining'); return
    if os.path.exists(f'{ddir}/DONE'):                              # cleared, but Drive still shows it: remove again
        try: os.remove(f'{ddir}/DONE')
        except OSError: pass
    os.makedirs(ddir, exist_ok=True)
    _retry(lambda: open(f'{ddir}/.write_test', 'w').write('ok'), 'first write to the Drive run folder')
    for old in glob.glob(f'{ddir}/epo*.tar'): os.remove(old)       # leftovers of an unfinished run: start clean
    local = f'/content/p1_runs/{label}'
    _copy_authors(folder, local)
    sp = f'{local}/model/pytorch/dcrnn_supervisor.py'; s = open(sp).read()
    assert s.count('random_seed = 0') == 1; open(sp, 'w').write(s.replace('random_seed = 0', f'random_seed = {seed}'))
    cp = f'{local}/data/model/dcrnn_la.yaml'; c = open(cp).read()
    assert 'dataset_dir: data/PEMSD8' in c; open(cp, 'w').write(c.replace('dataset_dir: data/PEMSD8', f'dataset_dir: {DATA8}'))
    stdout_path = f'/content/p1_runs/{label}.stdout.log'
    print(f'=== training {label} ({folder}, seed {seed}); about 1 h')

    def sync():
        for f in glob.glob(f'{local}/models/epo*.tar'):
            dst = f'{ddir}/{os.path.basename(f)}'
            if not os.path.exists(dst) or os.path.getsize(dst) != os.path.getsize(f): shutil.copy2(f, dst)
        for f in glob.glob(f'{local}/data/model/dcrnn_DR_*/info.log'): shutil.copy2(f, f'{ddir}/info.log')
        if os.path.exists(stdout_path): shutil.copy2(stdout_path, f'{ddir}/stdout.log')
    stop = threading.Event()
    def loop():
        fails = 0
        while not stop.wait(120):
            try:
                sync(); fails = 0
            except OSError as e:
                fails += 1
                if fails == 3: print(f'(note: copying to Drive has failed 3 times in a row; training continues, files are safe on local disk: {e})')
    threading.Thread(target=loop, daemon=True).start()

    t0 = time.time()
    with open(stdout_path, 'w') as log:
        p = subprocess.Popen([sys.executable, 'dcrnn_train_pytorch.py', '--config_filename=data/model/dcrnn_la.yaml'],
                             cwd=local, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            log.write(line); log.flush()
            if 'val_mae' in line or 'Traceback' in line or 'Error' in line:
                print(f'[{(time.time()-t0)/60:5.1f} min] ' + line.split(' - ')[-1].strip()[:150])
        p.wait()
    stop.set()
    if p.returncode != 0:
        print(open(stdout_path).read()[-4000:]); raise RuntimeError(f'{label} failed; paste the output above into Claude')
    _retry(sync, 'final copy of checkpoints and logs to Drive', tries=10, wait=30)
    best = max(_epochs(f'{local}/models'))
    info = dict(best_epoch=best, minutes=round((time.time()-t0)/60, 1), seed=seed, folder=folder)
    _retry(lambda: open(f'{ddir}/DONE', 'w').write(json.dumps(info)), 'writing the DONE marker')
    print(f'{label}: DONE, best epoch {best}, {info["minutes"]:.0f} min')

def evaluate_run(label):
    folder, seed = RUNS[label]; ddir = f'{RUNDR}/{label}'
    run_models = f'/content/p1_runs/{label}/models'            # present if this session trained the run
    try:
        best = json.load(open(f'{ddir}/DONE'))['best_epoch']
    except OSError:
        best = max(_epochs(run_models))
    src = f'{run_models}/epo{best}.tar'
    if not os.path.exists(src): src = f'{ddir}/epo{best}.tar'
    local = f'/content/p1_eval/{label}'
    _copy_authors(folder, local)
    # the PEMSD8 variants save to models/ but load from <folder>/models/ (hard-coded): put the checkpoint in both
    for mdir in [f'{local}/models', f'{local}/{folder}/models']:
        os.makedirs(mdir, exist_ok=True); shutil.copy2(src, f'{mdir}/epo{best}.tar')
    npz = f'{OUT8}/{label}.npz' if SAVE_PREDICTIONS else f'/content/{label}.npz'
    r = subprocess.run([sys.executable, f'{SCRIPTS}/eval_checkpoint.py', '--config', 'data/model/dcrnn_la.yaml',
                        '--epoch', str(best), '--data_dir', DATA8, '--out', npz, '--log_dir', '/content/p1_eval_logs'],
                       cwd=local, capture_output=True, text=True, env={**os.environ, 'SEED': str(seed)})
    print('\n'.join(l for l in r.stdout.splitlines() if 'authors-style' in l or l.startswith('saved')))
    if r.returncode != 0:
        print(r.stdout[-3000:]); print(r.stderr[-5000:]); raise RuntimeError(f'evaluation of {label} failed; paste the output above into Claude')
    z = np.load(npz); rows = []
    for split in ['val', 'test']:
        for pad in [None, 128]:
            for row in horizon_metrics.horizon_table(z[f'{split}_pred'], z[f'{split}_truth'], pad_to_batch=pad):
                if pad: row['view'] += '_padded'
                rows.append(dict(label=label, split=split, epoch=best, seed=seed, law=('underwood' if 'phy' in folder else 'none'), reg_lambda=(0.002 if 'phy' in folder else None), **row))
    df = pd.DataFrame(rows)
    _retry(lambda: df.to_csv(f'{RES}/pemsd8_runs/{label}.csv', index=False), 'writing the metrics CSV')
    print(f'{label}: evaluated at epoch {best} -> {RES}/pemsd8_runs/{label}.csv')
    return df

for label in MY_RUNS:
    train(label)
    evaluate_run(label)

=== training pemsd8_gpde_quantile_seed3 (gpde_quantile_model_PEMSD8, seed 3); about 1 h
[  1.1 min] Epoch [0/50] (98) val_mae:4.0651, val_mis: 93.4930, val_width: 5.8193, val_mse: 47.8890, val_rmse: 6.9202 ,
[  2.1 min] Epoch [1/50] (196) val_mae:3.7549, val_mis: 82.8304, val_width: 5.6515, val_mse: 41.0463, val_rmse: 6.4067 ,
[  3.1 min] Epoch [2/50] (294) val_mae:2.8602, val_mis: 60.6666, val_width: 4.9792, val_mse: 27.4330, val_rmse: 5.2377 ,
[  4.1 min] Epoch [3/50] (392) val_mae:3.9968, val_mis: 99.8237, val_width: 4.8784, val_mse: 38.8204, val_rmse: 6.2306 ,
[  5.1 min] Epoch [4/50] (490) val_mae:2.3163, val_mis: 46.3802, val_width: 4.5151, val_mse: 22.2978, val_rmse: 4.7221 ,
[  6.1 min] Epoch [5/50] (588) val_mae:2.4519, val_mis: 49.7270, val_width: 4.1813, val_mse: 22.6359, val_rmse: 4.7577 ,
[  7.1 min] Epoch [6/50] (686) val_mae:2.7431, val_mis: 58.8262, val_width: 4.2479, val_mse: 25.6610, val_rmse: 5.0657 ,
[  8.1 min] Epoch [7/50] (784) val_mae:2.3401, val_mis: 51.5158, v

## 5. Analysis (after all four runs are DONE; any account, no GPU needed)

In a fresh session run **sections 0 and 3** first, then this section.

- **A.** The rerun against Table 1 (averaged reading), next to the first run.
- **B. The seed check.** Phy minus GPDE per seed: the two new pairs (authors' own physics code), then all five pairs pooled, with a paired t-test.
- **C.** Plain-language verdict for the critique.
- **D.** Do the authors' physics code (seeds 3–4) and the re-implemented loss (seeds 0–2) give the same results?

In [14]:
sys.path.insert(0, SCRIPTS)
import importlib, horizon_metrics; importlib.reload(horizon_metrics)
import pandas as pd, numpy as np
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 40)

missing = [r for r in RUNS if not os.path.exists(f'{RES}/pemsd8_runs/{r}.csv')]
READY = not missing
if not READY:
    print('Waiting for runs that are not evaluated yet:', missing)
    print('Run sections 0, 3 and 5 again when all four are DONE.')
else:
    new = pd.concat([pd.read_csv(f'{RES}/pemsd8_runs/{r}.csv') for r in RUNS])
    new['model'] = new.label.map(lambda l: 'GPDE+Quantile+Phy' if '_phy_' in l else 'GPDE+Quantile')
    new['batch'] = 'rerun'
    # first run (read only from CS637): GPDE+Quantile seeds 0-2 (ours), Phy seeds 0-2 (re-implemented loss)
    old_g = pd.concat([pd.read_csv(f'{READ_BASE}/p1/results/pemsd8_runs/pemsd8_gpde_quantile_seed{k}.csv') for k in range(3)])
    old_g['model'] = 'GPDE+Quantile'
    p2 = pd.read_csv(f'{READ_BASE}/p2/results/horizon_metrics.csv')
    phy_labels = {'pemsd8_underwood_ours': 0, 'pemsd8_underwood_ours_seed1': 1, 'pemsd8_underwood_ours_seed2': 2}
    old_p = p2[p2.label.isin(phy_labels)].drop_duplicates(['label', 'split', 'view', 'minutes'], keep='last').copy()
    old_p['seed'] = old_p.label.map(phy_labels); old_p['model'] = 'GPDE+Quantile+Phy'
    old = pd.concat([old_g, old_p]); old['batch'] = 'first run'
    allr = pd.concat([new, old], ignore_index=True)
    allr = allr[(allr.split == 'test') & allr.view.isin(['step', 'avg'])]
    allr.to_csv(f'{RES}/pemsd8_rerun_all_runs_test.csv', index=False)
    print('seeds per model and batch:'); print(allr.groupby(['batch', 'model']).seed.unique())

seeds per model and batch:
batch      model            
first run  GPDE+Quantile        [0, 1, 2]
           GPDE+Quantile+Phy    [0, 2, 1]
rerun      GPDE+Quantile           [3, 4]
           GPDE+Quantile+Phy       [3, 4]
Name: seed, dtype: object


In [15]:
if not READY:
    print('Skipped: waiting for all four runs (see the message above).')
else:
    # A. Rerun vs Table 1, averaged reading (PEMSD8 columns: MAE, MIS, MSE, RMSE)
    T1_8 = {'GPDE+Quantile':     {15: (1.041, 12.633, 4.732, 2.175), 30: (1.251, 19.200, 7.730, 2.780), 60: (1.540, 29.760, 12.714, 3.565)},
            'GPDE+Quantile+Phy': {15: (1.024, 12.063, 4.676, 2.162), 30: (1.236, 18.497, 7.696, 2.774), 60: (1.521, 28.812, 12.524, 3.539)}}
    g = allr[allr.view == 'avg'].groupby(['model', 'batch', 'minutes'])
    A = g[['mae', 'mis_authors', 'rmse', 'coverage']].mean().join(g.mae.std().rename('mae_std')).join(g.seed.nunique().rename('n_seeds')).reset_index()
    A['T1_mae'] = [T1_8[m][mi][0] for m, mi in zip(A.model, A.minutes)]
    A['mae_diff_%'] = 100 * (A.mae - A.T1_mae) / A.T1_mae
    A['T1_mis'] = [T1_8[m][mi][1] for m, mi in zip(A.model, A.minutes)]
    A['mis_diff_%'] = 100 * (A.mis_authors - A.T1_mis) / A.T1_mis
    A = A.sort_values(['model', 'minutes', 'batch'])
    A.round(4).to_csv(f'{RES}/pemsd8_rerun_vs_table1.csv', index=False)
    display(A[['model', 'batch', 'minutes', 'n_seeds', 'mae', 'mae_std', 'T1_mae', 'mae_diff_%', 'mis_authors', 'T1_mis', 'mis_diff_%', 'rmse', 'coverage']].round(3))

,model,batch,minutes,n_seeds,mae,mae_std,T1_mae,mae_diff_%,mis_authors,T1_mis,mis_diff_%,rmse,coverage
0,GPDE+Quantile,first run,15,3,1.044,0.023,1.041,0.322,12.385,12.633,-1.965,2.172,0.873
3,GPDE+Quantile,rerun,15,2,1.050,0.041,1.041,0.825,12.695,12.633,0.491,2.173,0.873
1,GPDE+Quantile,first run,30,3,1.255,0.024,1.251,0.327,18.732,19.200,-2.439,2.778,0.804
4,GPDE+Quantile,rerun,30,2,1.260,0.039,1.251,0.723,19.525,19.200,1.692,2.789,0.794
2,GPDE+Quantile,first run,60,3,1.542,0.022,1.540,0.147,28.704,29.760,-3.549,3.566,0.724
5,GPDE+Quantile,rerun,60,2,1.553,0.047,1.540,0.817,30.512,29.760,2.527,3.605,0.701
6,GPDE+Quantile+Phy,first run,15,3,1.060,0.015,1.024,3.480,12.581,12.063,4.297,2.212,0.882
9,GPDE+Quantile+Phy,rerun,15,2,1.076,0.048,1.024,5.034,13.163,12.063,9.119,2.238,0.864
7,GPDE+Quantile+Phy,first run,30,3,1.268,0.016,1.236,2.621,19.060,18.497,3.045,2.805,0.807
10,GPDE+Quantile+Phy,rerun,30,2,1.286,0.043,1.236,4.014,19.899,18.497,7.582,2.832,0.783


In [16]:
if not READY:
    print('Skipped: waiting for all four runs (see the message above).')
else:
    # B. Seed check: Phy minus GPDE, paired by seed (single-horizon test metrics)
    from scipy import stats
    def paired(df):
        rows = []
        for mnt in [15, 30, 60]:
            for k in ['mae', 'rmse', 'mis_authors', 'mis_std', 'coverage']:
                sel = df[(df.view == 'step') & (df.minutes == mnt)]
                a = sel[sel.model == 'GPDE+Quantile'].set_index('seed')[k]
                b = sel[sel.model == 'GPDE+Quantile+Phy'].set_index('seed')[k]
                d = (b - a).dropna().sort_index()
                better = (d > 0) if k == 'coverage' else (d < 0)
                p = stats.ttest_rel(b[d.index], a[d.index]).pvalue if len(d) >= 3 else np.nan
                rows.append(dict(minutes=mnt, metric=k, n_pairs=len(d), gpde_mean=a.mean(), phy_mean=b.mean(),
                                 diff_mean=d.mean(), diff_pct=100 * d.mean() / a.mean(),
                                 per_seed=' / '.join(f'{s}:{x:+.3f}' for s, x in d.items()),
                                 phy_better_in=f'{int(better.sum())} of {len(d)}',
                                 gap_over_noise=abs(d.mean()) / max(a.std(), b.std()), paired_t_p=p))
        return pd.DataFrame(rows)
    B_new = paired(allr[allr.batch == 'rerun']);  B_new.round(4).to_csv(f'{RES}/pemsd8_rerun_seed_check_new_pairs.csv', index=False)
    B_all = paired(allr);                          B_all.round(4).to_csv(f'{RES}/pemsd8_rerun_seed_check_all_pairs.csv', index=False)
    print("New pairs only (seeds 3-4, authors' own physics code):"); display(B_new.drop(columns='paired_t_p').round(4))
    print('All five pairs (seeds 0-4):'); display(B_all.round(4))
    print('diff = Phy minus GPDE: lower is better for mae, rmse, mis; higher for coverage. '
          'gap_over_noise below 1 = within run-to-run noise. paired_t_p below 0.05 = unlikely to be noise.')

New pairs only (seeds 3-4, authors' own physics code):


,minutes,metric,n_pairs,gpde_mean,phy_mean,diff_mean,diff_pct,per_seed,phy_better_in,gap_over_noise
0,15,mae,2,1.2365,1.2622,0.0258,2.0852,3:+0.029 / 4:+0.022,0 of 2,0.6628
1,15,rmse,2,2.6308,2.6825,0.0517,1.9643,3:+0.087 / 4:+0.017,0 of 2,0.5194
2,15,mis_authors,2,17.7546,18.2162,0.4616,2.5999,3:+1.107 / 4:-0.183,1 of 2,0.3459
3,15,mis_std,2,16.5181,16.9540,0.4358,2.6384,3:+1.078 / 4:-0.206,1 of 2,0.3364
4,15,coverage,2,0.7986,0.7852,-0.0133,-1.6692,3:-0.040 / 4:+0.013,1 of 2,0.5538
5,30,mae,2,1.5706,1.5953,0.0247,1.5720,3:+0.023 / 4:+0.027,0 of 2,0.6184
6,30,rmse,2,3.5581,3.5790,0.0208,0.5859,3:+0.015 / 4:+0.027,0 of 2,0.2508
7,30,mis_authors,2,30.2457,30.5053,0.2596,0.8582,3:+0.753 / 4:-0.233,1 of 2,0.1614
8,30,mis_std,2,28.6752,28.9100,0.2349,0.8191,3:+0.730 / 4:-0.260,1 of 2,0.1495
9,30,coverage,2,0.6831,0.6672,-0.0159,-2.3262,3:-0.038 / 4:+0.006,1 of 2,0.7937


All five pairs (seeds 0-4):


,minutes,metric,n_pairs,gpde_mean,phy_mean,diff_mean,diff_pct,per_seed,phy_better_in,gap_over_noise,paired_t_p
0,15,mae,5,1.2350,1.2545,0.0194,1.5732,0:+0.026 / 1:+0.031 / 2:-0.012 / 3:+0.029 / 4:...,1 of 5,0.7873,0.0717
1,15,rmse,5,2.6282,2.6721,0.0439,1.6696,0:+0.058 / 1:+0.050 / 2:+0.007 / 3:+0.087 / 4:...,0 of 5,0.8233,0.0382
2,15,mis_authors,5,17.3989,17.8035,0.4046,2.3255,0:+0.148 / 1:+1.234 / 2:-0.283 / 3:+1.107 / 4:...,2 of 5,0.4636,0.2764
3,15,mis_std,5,16.1639,16.5491,0.3852,2.3828,0:+0.122 / 1:+1.203 / 2:-0.271 / 3:+1.078 / 4:...,2 of 5,0.4535,0.2898
4,15,coverage,5,0.8029,0.7999,-0.0030,-0.3707,0:+0.020 / 1:-0.022 / 2:+0.014 / 3:-0.040 / 4:...,3 of 5,0.1604,0.8138
5,30,mae,5,1.5664,1.5826,0.0162,1.0334,0:+0.019 / 1:+0.033 / 2:-0.021 / 3:+0.023 / 4:...,1 of 5,0.5974,0.1676
6,30,rmse,5,3.5424,3.5578,0.0154,0.4358,0:+0.005 / 1:+0.034 / 2:-0.004 / 3:+0.015 / 4:...,1 of 5,0.3382,0.0904
7,30,mis_authors,5,29.2665,29.6975,0.4310,1.4728,0:+0.380 / 1:+1.188 / 2:+0.068 / 3:+0.753 / 4:...,1 of 5,0.3911,0.1600
8,30,mis_std,5,27.7001,28.1149,0.4148,1.4975,0:+0.361 / 1:+1.154 / 2:+0.089 / 3:+0.730 / 4:...,1 of 5,0.3829,0.1670
9,30,coverage,5,0.6985,0.6877,-0.0107,-1.5327,0:-0.004 / 1:-0.015 / 2:-0.002 / 3:-0.038 / 4:...,1 of 5,0.4289,0.2299


diff = Phy minus GPDE: lower is better for mae, rmse, mis; higher for coverage. gap_over_noise below 1 = within run-to-run noise. paired_t_p below 0.05 = unlikely to be noise.


In [17]:
if not READY:
    print('Skipped: waiting for all four runs (see the message above).')
else:
    # C. Verdict lines for the critique (single-horizon MAE and MIS, all five pairs)
    for mnt in [15, 30, 60]:
        for k, name in [('mae', 'MAE'), ('mis_authors', 'MIS')]:
            r = B_all[(B_all.minutes == mnt) & (B_all.metric == k)].iloc[0]
            verdict = ('within noise' if (r.gap_over_noise < 1 or not r.paired_t_p < 0.05)
                       else ('physics better' if r.diff_mean < 0 else 'physics worse'))
            print(f'{mnt:2d} min {name}: GPDE {r.gpde_mean:.3f} -> +Phy {r.phy_mean:.3f} ({r.diff_pct:+.1f}%), '
                  f'physics better in {r.phy_better_in} seeds, paired t-test p = {r.paired_t_p:.2f} -> {verdict}')
    a15 = allr[(allr.view == 'avg') & (allr.minutes == 15)].groupby('model').mae
    print(f"\nPaper (Table 1, averaged 15 min): 1.041 -> 1.024 (gain 0.017). "
          f"Ours, all seeds: {a15.mean()['GPDE+Quantile']:.3f} -> {a15.mean()['GPDE+Quantile+Phy']:.3f}; "
          f"seed std {a15.std()['GPDE+Quantile']:.3f} (GPDE) / {a15.std()['GPDE+Quantile+Phy']:.3f} (Phy)")

15 min MAE: GPDE 1.235 -> +Phy 1.254 (+1.6%), physics better in 1 of 5 seeds, paired t-test p = 0.07 -> within noise
15 min MIS: GPDE 17.399 -> +Phy 17.804 (+2.3%), physics better in 2 of 5 seeds, paired t-test p = 0.28 -> within noise
30 min MAE: GPDE 1.566 -> +Phy 1.583 (+1.0%), physics better in 1 of 5 seeds, paired t-test p = 0.17 -> within noise
30 min MIS: GPDE 29.266 -> +Phy 29.697 (+1.5%), physics better in 1 of 5 seeds, paired t-test p = 0.16 -> within noise
60 min MAE: GPDE 2.009 -> +Phy 2.027 (+0.9%), physics better in 1 of 5 seeds, paired t-test p = 0.30 -> within noise
60 min MIS: GPDE 46.680 -> +Phy 47.832 (+2.5%), physics better in 0 of 5 seeds, paired t-test p = 0.03 -> within noise

Paper (Table 1, averaged 15 min): 1.041 -> 1.024 (gain 0.017). Ours, all seeds: 1.046 -> 1.066; seed std 0.026 (GPDE) / 0.028 (Phy)


In [18]:
if not READY:
    print('Skipped: waiting for all four runs (see the message above).')
else:
    # D. Authors' physics code (rerun, seeds 3-4) vs re-implemented loss (first run, seeds 0-2): single-horizon MAE and coverage
    phy = allr[(allr.model == 'GPDE+Quantile+Phy') & (allr.view == 'step')]
    D = phy.groupby(['batch', 'minutes']).agg(mae_mean=('mae', 'mean'), mae_min=('mae', 'min'), mae_max=('mae', 'max'),
                                              coverage_mean=('coverage', 'mean'), seeds=('seed', lambda s: sorted(s.unique()))).reset_index()
    D.round(4).to_csv(f'{RES}/pemsd8_rerun_phy_code_check.csv', index=False)
    display(D.round(3))
    print("If the rerun's MAE range overlaps the first run's, the two loss implementations behave the same.")

,batch,minutes,mae_mean,mae_min,mae_max,coverage_mean,seeds
0,first run,15,1.249,1.234,1.268,0.810,"[0, 1, 2]"
1,first run,30,1.574,1.560,1.592,0.701,"[0, 1, 2]"
2,first run,60,2.017,1.992,2.036,0.576,"[0, 1, 2]"
3,rerun,15,1.262,1.235,1.290,0.785,"[3, 4]"
4,rerun,30,1.595,1.569,1.622,0.667,"[3, 4]"
5,rerun,60,2.041,1.986,2.096,0.531,"[3, 4]"


If the rerun's MAE range overlaps the first run's, the two loss implementations behave the same.
